# PatchPilot — 02 · Patch generator: QLoRA fine-tuning of Gemma 4 E4B (A2)

Teaches `google/gemma-4-E4B-it` to answer the agent's repair prompt with correct **SEARCH/REPLACE**
edits. Each training chat is exactly what the agent sends (system prompt, bug report, the tests the
fix added, the trimmed source file) followed by the developers' fix rewritten as edit blocks.

**Runtime:** Kaggle notebook · **GPU T4 x1** · **Internet on** · secret **`HF_TOKEN`** (write token).
Expected run: training plus a merge and GGUF conversion; the session can take several hours.

**Outputs pushed to the Hub:**
* `HF_ADAPTER`: the LoRA adapter, `training_log.json`, `loss_curve.png`, peak GPU memory, model card;
* `HF_GGUF`: the merged model converted to GGUF **Q4_K_M** for Ollama on the laptop.

In [ ]:
# ---- Configuration: set HF_USER; everything else can stay ----
HF_USER = "YOUR_HF_USERNAME"
HF_DATASET = f"{HF_USER}/patchpilot-patchgen"
HF_ADAPTER = f"{HF_USER}/patchpilot-gemma4-e4b-lora"
HF_GGUF = f"{HF_USER}/patchpilot-gemma4-e4b-gguf"

BASE_MODEL = "google/gemma-4-E4B-it"
MAX_LEN = 3072  # tokens; the dataset keeps chats under ~2,800 tokens
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EPOCHS = 1
LR = 2e-4
BATCH = 1
GRAD_ACCUM = 16
SEED = 42

In [ ]:
%%capture
!pip install -q "transformers>=4.56" "peft>=0.17" "trl>=0.23" "bitsandbytes>=0.46" "accelerate>=1.10" "datasets>=3.0" "huggingface_hub>=0.34" matplotlib

In [ ]:
import json
import os
import random

import numpy as np
import torch
from huggingface_hub import HfApi, login

try:  # Kaggle
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:  # Colab
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Turn on the GPU accelerator in the notebook settings"
print("GPU:", torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
from datasets import load_dataset

raw = load_dataset(HF_DATASET, data_files={"train": "train.jsonl.gz", "validation": "val.jsonl.gz"})


def to_prompt_completion(example):
    # TRL computes the loss on the completion only (the assistant's edit blocks).
    return {"prompt": example["messages"][:-1], "completion": example["messages"][-1:]}


data = raw.map(to_prompt_completion, remove_columns=raw["train"].column_names)
print(data)

In [ ]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant,
    torch_dtype=compute_dtype,
    device_map={"": 0},
    attn_implementation="eager",
)
lora = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
print("model loaded; GPU memory (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

In [ ]:
from trl import SFTConfig, SFTTrainer

config = SFTConfig(
    output_dir="ckpt",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=1,
    gradient_checkpointing=True,
    max_length=MAX_LEN,
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    completion_only_loss=True,
    optim="paged_adamw_8bit",
)
trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    processing_class=tokenizer,
    peft_config=lora,
)
trainer.model.print_trainable_parameters()
result = trainer.train()
peak_gb = round(torch.cuda.max_memory_allocated() / 1e9, 2)
print(result)
print("peak GPU memory (GB):", peak_gb)
log = {
    "log_history": trainer.state.log_history,
    "peak_gpu_memory_gb": peak_gb,
    "train_examples": len(data["train"]),
    "val_examples": len(data["validation"]),
}
with open("training_log.json", "w") as f:
    json.dump(log, f, indent=1)

In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
train = [(h["step"], h["loss"]) for h in hist if "loss" in h]
val = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(6, 3.5))
plt.plot(*zip(*train), label="train")
if val:
    plt.plot(*zip(*val), marker="o", label="validation")
plt.xlabel("optimizer step")
plt.ylabel("loss")
plt.title("Gemma 4 E4B QLoRA fine-tuning")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()

In [ ]:
card = f"""---
license: apache-2.0
base_model: {BASE_MODEL}
library_name: peft
tags: [lora, qlora, program-repair, code, patchpilot]
datasets: [{HF_DATASET}]
---
# PatchPilot patch generator: Gemma 4 E4B QLoRA adapter

LoRA adapter for `{BASE_MODEL}` that answers PatchPilot's repair prompt with SEARCH/REPLACE edit
blocks. Trained on `{HF_DATASET}` (SWE-bench train split; repositories disjoint from SWE-bench Lite,
see `results/leakage_check.json` in the PatchPilot repository).

* QLoRA: 4-bit NF4 base, LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}, all attention and MLP projections.
* {len(data["train"])} training chats, {EPOCHS} epoch(s), lr {LR}, effective batch {BATCH * GRAD_ACCUM},
  max length {MAX_LEN}, seed {SEED}, loss on the assistant answer only.
* Peak GPU memory: {peak_gb} GB on {torch.cuda.get_device_name(0)}.
* Evaluation: PatchPilot experiments E2-E5, results in the PatchPilot repository.
* Limitations: trained on single-file fixes with at most three hunks; English issues; Python only.
"""
with open("README.md", "w") as f:
    f.write(card)
trainer.model.save_pretrained("adapter")
tokenizer.save_pretrained("adapter")
for name in ["README.md", "training_log.json", "loss_curve.png"]:
    os.replace(name, f"adapter/{name}")
api = HfApi()
api.create_repo(HF_ADAPTER, exist_ok=True)
api.upload_folder(
    folder_path="adapter", repo_id=HF_ADAPTER, commit_message="PatchPilot QLoRA adapter"
)
print("adapter pushed to https://huggingface.co/" + HF_ADAPTER)

## Merge and convert to GGUF for Ollama

The adapter is merged into the full-precision base model on the CPU (Kaggle has ~30 GB RAM), then
converted with llama.cpp and quantised to **Q4_K_M**, the same quantisation level as the
`gemma4:e4b` model PatchPilot uses as its baseline, so both sit in a similar ~6 GB on disk.

In [ ]:
import gc

del trainer, model
gc.collect()
torch.cuda.empty_cache()

from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"": "cpu"}
)
merged = PeftModel.from_pretrained(base, "adapter").merge_and_unload()
merged.save_pretrained("merged", safe_serialization=True)
tokenizer.save_pretrained("merged")
del base, merged
gc.collect()
print("merged model saved")

In [ ]:
%%bash
set -e
git clone --depth 1 https://github.com/ggml-org/llama.cpp
pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
python llama.cpp/convert_hf_to_gguf.py merged --outtype f16 --outfile patchpilot-gemma4-e4b-f16.gguf
cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
llama.cpp/build/bin/llama-quantize patchpilot-gemma4-e4b-f16.gguf patchpilot-gemma4-e4b-Q4_K_M.gguf Q4_K_M
ls -lh *.gguf

In [ ]:
api.create_repo(HF_GGUF, exist_ok=True)
api.upload_file(
    path_or_fileobj="patchpilot-gemma4-e4b-Q4_K_M.gguf",
    path_in_repo="patchpilot-gemma4-e4b-Q4_K_M.gguf",
    repo_id=HF_GGUF,
    commit_message="Merged PatchPilot Gemma 4 E4B, GGUF Q4_K_M",
)
print("GGUF pushed to https://huggingface.co/" + HF_GGUF)